# Consolidação das unidades — os dois baldes, uma triagem

> **Códigos e siglas** (`U_…`, Ajuste N, [conferido]): o que cada um quer dizer está no [glossário](../../glossario.md).

**Notebook genérico:** roda sobre qualquer extração apontada por `TRACE` em `base_pipeline.py`. Só contagens e nomes
de unidade, papel e mês; nada de `exec_id` nem texto de caso.

**Onde este notebook fica no desenho** (`../../plano-atual.md` §1): um funil separa os steps em dois baldes, cada um
classificado à parte com o mesmo catálogo de unidades; aqui os dois se juntam.

| Balde | Notebook | Entrega |
|---|---|---|
| visível — erro com exceção | `analise_trace_esteira_juridica.ipynb` | erros com unidade e ocorrência (cascata × unidade) |
| invisível — falha de ferramenta devolvida como texto | `falhas_silenciosas.ipynb` (§13.8) | falhas com unidade (`REGRAS_INVISIVEL`) e ocorrência pela mesma régua |
| **consolidação** | **este (§14.x)** | **a triagem final das unidades, com a recorrência contada sobre a união** |

**As regras da consolidação** (Ajuste 12, 04/10; livro-razão):

1. **Junta ocorrências, não totais.** Execuções, meses e papéis de uma unidade são contados sobre a união dos dois
   canais — a mesma execução pode ter a unidade nos dois.
2. **A mesma régua de ocorrência nos dois canais** (decisão do Rafael, 02/10): cascata de steps consecutivos do mesmo
   papel × unidade.
3. **Erros e tokens continuam do visível.** O custo de uma falha silenciosa é retrabalho, medido no notebook dela; somar
   tokens de steps que não erraram misturaria duas medidas.
4. **A decisão (candidata, fora, sinal de harness…) é a mesma função `triagem()`** do visível, com as ocorrências
   silenciosas como parâmetro. Sem elas, a saída é a da esteira — a §14.3 confere.

Docs: o método erro → memória e esta etapa final, [`../docs/09-metodologia-erro-a-memoria.md`](../docs/09-metodologia-erro-a-memoria.md);
o balde invisível, [`../docs/13-racionais-falhas-silenciosas.md`](../docs/13-racionais-falhas-silenciosas.md). Seções
§14.x (cada notebook é dono de um intervalo; README da pasta).

In [1]:
import os
import pandas as pd
from base_pipeline import *   # espinha compartilhada desta pasta — ver base_pipeline.py

B = carregar_base()                              # B.df, B.EU (o balde visível), ...
F = falhas_silenciosas(B.df)
sil = F[F["falha"] == "silenciosa"]
OV = ocorrencias_visiveis(B.EU)                  # balde visível, formato comum
OS = ocorrencias_silenciosas(sil, formas_das_falhas(B.df, sil))   # balde invisível, formato comum
print(f"{BASE_ID}: visível {len(OV)} erros em {OV['ocorrencia'].nunique()} ocorrências · "
      f"invisível {len(OS)} steps, {OS['unidade'].notna().sum()} com unidade do catálogo")

base1: visível 498 erros em 437 ocorrências · invisível 119 steps, 6 com unidade do catálogo


## 14.1 · As duas tabelas de ocorrências

Conferências antes de juntar:

- **nenhum step nos dois canais** (o funil separa por step; tem de dar 0);
- **execuções em comum** por unidade entre os canais — se houver, a união conta a execução uma vez só (regra 1).

In [2]:
chave = ["exec_id", "role", "idx"]
print(f"steps nos dois canais: {len(OV[chave].merge(OS[chave], on=chave))}  (tem de ser 0)")
com_u = OS[OS["unidade"].notna()]
linhas = []
for u in sorted(set(com_u["unidade"])):
    v, s = OV[OV["unidade"] == u], com_u[com_u["unidade"] == u]
    linhas.append({"unidade": u, "erros visíveis": len(v), "ocorrências visíveis": v["ocorrencia"].nunique(),
                   "steps silenciosos": len(s), "ocorrências silenciosas": s["ocorrencia"].nunique(),
                   "execuções em comum": len(set(v["exec_id"]) & set(s["exec_id"]))})
display(pd.DataFrame(linhas).set_index("unidade"))

steps nos dois canais: 0  (tem de ser 0)


,erros visíveis,ocorrências visíveis,steps silenciosos,ocorrências silenciosas,execuções em comum
unidade,,,,,
U_repr_colado,6,6,6,6,0


## 14.2 · A triagem consolidada

A mesma régua da esteira (§9.2): ≥3 execuções e ≥2 meses para candidata; sensibilidade com ≥5 e ≥3. As colunas
`ocorrências visíveis` / `ocorrências silenciosas` mostram de onde vem cada ocorrência; `erros` e `tokens` são do
visível.

In [3]:
TC = triagem(B.EU, silenciosas=OS)
TCS = triagem(B.EU, 5, 3, silenciosas=OS)
cand = TC[TC["decisão"] == "candidato"]
limitrofes = sorted(set(cand["nome"]) - set(TCS.loc[TCS["decisão"] == "candidato", "nome"]))
print(f"{len(TC)} unidades: {len(cand)} candidatas, {(TC['decisão'] == SINAL_HARNESS).sum()} sinal de harness, "
      f"{(TC['decisão'] == INVESTIGAR_CRITICO).sum()} erro crítico, {(TC['decisão'] == 'não-memória').sum()} não-memória, "
      f"{TC['decisão'].str.startswith('revisar').sum()} a revisar, {TC['decisão'].str.startswith('fora').sum()} fora")
print(f"sensibilidade (≥5 execuções e ≥3 meses) — deixam de ser candidatas: {limitrofes or 'nenhuma'}")
display(TC[["unidade", "nome", "decisão", "ocorrências", "ocorrências visíveis", "ocorrências silenciosas", "erros",
            "execuções", "meses", "papéis", "tokens"]].set_index("unidade"))

15 unidades: 10 candidatas, 1 sinal de harness, 0 erro crítico, 2 não-memória, 2 a revisar, 0 fora
sensibilidade (≥5 execuções e ≥3 meses) — deixam de ser candidatas: ['Após step com erro, o que ele definiria não existe']


,nome,decisão,ocorrências,ocorrências visíveis,ocorrências silenciosas,erros,execuções,meses,papéis,tokens
unidade,,,,,,,,,,
U_texto_literal,Texto longo nunca dentro de literal de string,candidato,169,169,0,182,145,10,5,4004114
U_contrato_dict,Retorno das ferramentas de documento é dict,candidato,87,87,0,96,87,8,1,2989426
U_tipo_retorno,Retorno pode chegar como string,candidato,42,42,0,47,36,9,7,2106255
U_texto_solto,Explicação nunca solta no bloco de código,candidato,12,12,0,33,12,5,3,1388050
U_sandbox,Inventário do sandbox,candidato,17,17,0,17,17,8,6,887853
U_arg_nomeado,Ferramentas só aceitam argumento nomeado,candidato,25,25,0,36,23,11,7,394847
U_next_gerador,next() sobre expressão geradora falha no sandbox,candidato,11,11,0,13,11,8,2,339280
U_nome_inventado,Nome usado sem ter sido definido,candidato,5,5,0,5,5,4,2,294106
U_estado_perdido,"Após step com erro, o que ele definiria não ex...",candidato,5,5,0,5,4,3,2,213853


## 14.3 · O que a consolidação muda — comparada com a triagem só do visível

Duas conferências: (a) sem as silenciosas, a `triagem()` daqui é a da esteira (§9.2); (b) com elas, a lista de
células que mudam. Esperado na base 1: só a `U_repr_colado` (ocorrências e execuções 6 → 12); na base 2: 18 → 104.

In [4]:
TV = triagem(B.EU)
cols = [c for c in TV.columns]
a = TV.set_index("unidade")[cols[1:]]
b = TC.set_index("unidade")[cols[1:]].reindex(a.index)
mud = [{"unidade": u, "coluna": c, "só visível": a.at[u, c], "consolidada": b.at[u, c]}
       for u in a.index for c in a.columns if str(a.at[u, c]) != str(b.at[u, c])]
novas = sorted(set(TC["unidade"]) - set(TV["unidade"]))
print(f"unidades que só existem no canal silencioso: {novas or 'nenhuma'}")
print(f"decisões que mudam: {[m['unidade'] for m in mud if m['coluna'] == 'decisão'] or 'nenhuma'}")
display(pd.DataFrame(mud) if mud else "nenhuma célula muda")

unidades que só existem no canal silencioso: nenhuma
decisões que mudam: nenhuma


,unidade,coluna,só visível,consolidada
0,U_repr_colado,ocorrências,6,12
1,U_repr_colado,execuções,6,12


## 14.4 · A unidade consolidada — `U_repr_colado`, os dois canais

O mesmo gesto — colar o print de um retorno em vez de passar a variável — aparece de dois jeitos: quando a colagem quebra
a sintaxe, vira `SyntaxError` (canal visível); quando é Python válido dentro de `str()`, vira JSON inválido na ferramenta
(canal silencioso). A lição cobre os dois.

In [5]:
u = "U_repr_colado"
J = pd.concat([OV[OV["unidade"] == u], OS[OS["unidade"] == u]])
display(J.groupby(["canal", "role", "mes"]).agg(ocorrências=("ocorrencia", "nunique"),
                                                 execuções=("exec_id", "nunique")))
print("lição:", UNI[u][2])

ocorrências  execuções
canal      role              mes                            
silencioso RoteadorCivel     2026-05            3          3
                             2026-06            3          3
visível    ConversationAgent 2026-03            2          2
                             2026-05            1          1
           RoteadorCivel     2026-06            2          2
                             2026-07            1          1

lição: Referenciar a variável que guardou o retorno em vez de colar o print dele (truncado ou inteiro) dentro do código; se a ferramenta pede JSON, converter a variável com json.dumps(...), nunca com str(...).


## 14.5 · O que ficou sem unidade — a fila de trabalho do balde invisível

Falhas silenciosas sem regra para o catálogo. Não entram na triagem: são, para o balde invisível, o que o resíduo é para
o visível. Os grupos `sem_resultado` e `fora_da_cobertura` não são falha real (doc 13 §3); os outros pedem regra nova,
escrita olhando as duas bases.

In [6]:
display(OS[OS["unidade"].isna()].groupby("grupo").agg(steps=("idx", "size"), ocorrências=("ocorrencia", "nunique"),
                                                      execuções=("exec_id", "nunique"), papéis=("role", "nunique"))
          .sort_values("steps", ascending=False))

,steps,ocorrências,execuções,papéis
grupo,,,,
sem_resultado,43,31,30,4
argumento_do_agente,41,39,38,3
plataforma,18,16,13,5
nao_reconhecido,9,9,8,4
fora_da_cobertura,2,2,2,2


## 14.6 · Exportação

`resultados/triagem_consolidada.csv` — a triagem final, uma linha por unidade. Até o plano 4.2c (a triagem sai da
esteira e passa a morar aqui), o `resultados/candidatos_memoria.csv` da esteira continua existindo e é o balde visível
sozinho.

In [7]:
os.makedirs("resultados", exist_ok=True)
TC.to_csv(os.path.join("resultados", "triagem_consolidada.csv"), index=False)
print(f"{len(TC)} unidades → resultados/triagem_consolidada.csv")

15 unidades → resultados/triagem_consolidada.csv
